In [2]:
import numpy as np

fruits = np.load("../Data/fruits.npy")
fruits.shape

(300, 100, 100)

In [3]:
target = np.concatenate(
    [
        np.zeros(100), # Apple
        np.ones(100), # pine apple
        np.full(100,2) #banana
    ]
)
target.shape

(300,)

In [4]:
target

array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 2., 2., 2., 2.,
       2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2.,
       2., 2., 2., 2., 2.

#### train 과 test

In [5]:
train = fruits.reshape(-1,100,100,1)

from sklearn.model_selection import train_test_split

train_data, test_data, train_target, test_target = train_test_split(
    train,
    target,
    test_size=0.2,
    stratify=target,
    random_state=42
)
    

In [13]:
print(train_data.shape)
print(test_data.shape)
print(train_target.shape)
print(test_target.shape)

(240, 100, 100, 1)
(60, 100, 100, 1)
(240,)
(60,)


 #### CNN 만들기

In [14]:
train_scaled = train_data / 255.0
test_scaled = test_data / 255.0

In [15]:
from tensorflow import keras
from tensorflow.keras import layers

model = keras.Sequential()

# 1번째 합성곱층
model.add(layers.Conv2D(
    32,
    kernel_size=3,
    activation='relu',
    padding='same',
    input_shape=(100, 100, 1)
))

# 1번째 풀링층
model.add(layers.MaxPooling2D(2))

# 2번째 합성곱층
model.add(layers.Conv2D(
    64,
    kernel_size=3,
    activation='relu',
    padding='same'
))

# 2번째 풀링층
model.add(layers.MaxPooling2D(2))

# 1차원으로 펼치기 (입력층)
model.add(layers.Flatten())

# Dense층 (은닉층)
model.add(layers.Dense(100, activation='relu'))

# 과적합 방지(drop out 층)
model.add(layers.Dropout(0.4))

# 출력층: 과일 종류 3개
model.add(layers.Dense(3, activation='softmax'))

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d_2 (Conv2D)                    │ (None, 100, 100, 32)        │             320 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_2 (MaxPooling2D)       │ (None, 50, 50, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_3 (Conv2D)                    │ (None, 50, 50, 64)          │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_3 (MaxPooling2D)       │ (None, 25, 25, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten_1 (Flatten)                  │ (None, 40000)               │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 100)                 │       4,000,100 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 100)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_3 (Dense)                      │ (None, 3)                   │             303 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 4,019,219 (15.33 MB)

 Trainable params: 4,019,219 (15.33 MB)

 Non-trainable params: 0 (0.00 B)

####
입력 이미지: 100 × 100 × 1
↓
Conv2D: 이미지 특징 찾기
↓
MaxPooling2D: 중요한 특징만 남기고 크기 줄이기
↓
Conv2D: 더 복잡한 특징 찾기
↓
MaxPooling2D: 다시 크기 줄이기
↓
Flatten: 1차원으로 펼치기
↓
Dense: 특징을 보고 판단
↓
Softmax: 사과/파인애플/바나나 확률 출력

In [16]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [17]:
history = model.fit(
    train_scaled,
    train_target,
    epochs=20,
    validation_data=(test_scaled, test_target)
)

Epoch 1/20
8/8 ━━━━━━━━━━━━━━━━━━━━ 2s 81ms/step - accuracy: 0.3125 - loss: 1.3248 - val_accuracy: 0.3333 - val_loss: 1.1048
Epoch 2/20
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step - accuracy: 0.3167 - loss: 1.1042 - val_accuracy: 0.3167 - val_loss: 1.0986
Epoch 3/20
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step - accuracy: 0.3583 - loss: 1.0983 - val_accuracy: 0.3167 - val_loss: 1.0989
Epoch 4/20
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step - accuracy: 0.4875 - loss: 1.0941 - val_accuracy: 0.3500 - val_loss: 1.0988
Epoch 5/20
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step - accuracy: 0.4167 - loss: 1.0816 - val_accuracy: 0.2333 - val_loss: 1.1004
Epoch 6/20
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 62ms/step - accuracy: 0.3875 - loss: 1.0749 - val_accuracy: 0.3333 - val_loss: 1.1746
Epoch 7/20
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 64ms/step - accuracy: 0.3958 - loss: 1.1154 - val_accuracy: 0.3500 - val_loss: 1.1013
Epoch 8/20
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step - accuracy: 0.4417 - loss: 1.0689 - val_accuracy: 0.2833 - val_loss: 1.1253


In [18]:
model.evaluate(test_scaled, test_target)

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.3000 - loss: 1.1668


[1.16680109500885, 0.30000001192092896]